In [ ]:
!uv add onnxruntime==1.23.2 tokenizers numpy tqdm minsearch gitsource

⠙ sympy==1.14.0                                                                 

In [ ]:
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
!wget $PREFIX/download.py
!wget $PREFIX/embedder.py

In [ ]:
!uv run python download.py

In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)


In [2]:
v1[0]

np.float64(-0.02058200593003704)

## Q1. Loading the data

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
print(f"documents size {len(documents)}")
print(documents[0])

documents size 72
{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour

## Q2. Cosine similarity

In [6]:
c_sample = next(
    (d['content'] for d in documents if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md'), 
    None
)
print(c_sample) 

# Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over every document, so it takes a minute on our dataset.
Keeping everything in memory is fine here, but a larger dataset would
need too much space.

The third problem is brute-force search. For every query we compare the
query vector against every single document. With 1,000 documents this is
fine, probably even faster than anything smarter. But as the dataset
grows past 10,000 or so, it gets slow, and we'll want an approximate
method instead.

What we've done so far is exact nearest neighbor (NN) sea

In [7]:
c_vector = embed.encode(c_sample)
print(c_vector.dot(v1))

0.3610702814461231


## Q3. Chunking and search by hand

In [8]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
len(chunks)

295

In [12]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [23]:
import numpy as np

texts = [c['content'] for c in chunks] 
filenames = [c['filename'] for c in chunks]
X = np.array(embed.encode_batch(texts)) 

scores = X.dot(v1)
highest_score_idx = np.argmax(scores)

In [24]:
print(f"Highest Score: {scores[highest_score_idx]}")
print(f"Belongs to file: {filenames[highest_score_idx]}")

Highest Score: 0.6489015778372396
Belongs to file: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector search with minsearch

In [37]:
# Creating the index
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [38]:
query = "What metric do we use to evaluate a search engine?"
v = embed.encode(query)

In [30]:
results = vindex.search(v, num_results=5)
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

## Q5. Text search vs vector search

In [34]:
# Creating the index
from minsearch import Index

tindex = Index(text_fields=["content"],keyword_fields=["filename"])
tindex.fit(chunks)

In [36]:
results = tindex.search(query, num_results=5)
results[0]

{'start': 0,
 'content': '# Search\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\n## Search basics\n\nAt its core, every search engine does the same thing. It takes a query,\nscores every document for similarity, and returns the top results.\n\nFormally, there is a similarity function:\n\n```python\nscore = sim(query, document)\n```\n\nFor each document in the database, you compute this score. Then you\nrank all documents by score and return the top N. What makes a search\nengine different from another search engine is what `sim` actually\ncomputes.\n\n- text/lexical search (covered in this section): `sim` counts how\n  many words the query and the document share. It looks at the surface\n  form, the actual words, and matches them exactly.\n\n- vector/semantic search (covered in [module 2](../../02-vector-search/)):\n  `sim` compares the meaning of the query and the document. Same\n  function, different similarity m

In [39]:
q2 = "How do I store vectors in PostgreSQL?"
v2 = embed.encode(q2)

In [46]:
v2_results= vindex.search(v2,num_results=5)
t2_results = tindex.search(q2,num_results=5)

print("VECTOR SEARCH RESULTS")
for rank, r in enumerate(v2_results, 1):
    print(f" {rank}. {r['filename']}")

print("\nKEYWORD SEARCH RESULTS")
for rank, r in enumerate(t2_results, 1):
    print(f" {rank}. {r['filename']}") 


VECTOR SEARCH RESULTS
 1. 02-vector-search/lessons/08-pgvector.md
 2. 02-vector-search/lessons/08-pgvector.md
 3. 03-orchestration/lessons/05-rag.md
 4. 02-vector-search/lessons/08-pgvector.md
 5. 02-vector-search/lessons/08-pgvector.md

KEYWORD SEARCH RESULTS
 1. 02-vector-search/lessons/02-embeddings.md
 2. 03-orchestration/lessons/05-rag.md
 3. 02-vector-search/lessons/01-intro.md
 4. 03-orchestration/lessons/05-rag.md
 5. 02-vector-search/lessons/01-intro.md


02-vector-search/lessons/08-pgvector.md shows up in Vector Results but not in text results.

## Q6. Hybrid search

- Reciprocal Rank Fusion (RRF)
- RRF(d) = sum over lists of  1 / (k + rank(d))

In [47]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [57]:
q3 = "How do I give the model access to tools?"
v3 = embed.encode(q3)
v3_results= vindex.search(v3,num_results=5)
t3_results = tindex.search(q3,num_results=5)

In [59]:
h3_results = rrf([v3_results, t3_results])

print("VECTOR SEARCH RESULTS")
for rank, r in enumerate(v3_results, 1):
    print(f" {rank}. {r['filename']}")

print("\nKEYWORD SEARCH RESULTS")
for rank, r in enumerate(t3_results, 1):
    print(f" {rank}. {r['filename']}") 
    
print("\nHybrid SEARCH RESULTS")
for rank, r in enumerate(h3_results, 1):
    print(f" {rank}. {r['filename']}")

VECTOR SEARCH RESULTS
 1. 01-agentic-rag/lessons/01-intro.md
 2. 04-evaluation/lessons/02-ground-truth.md
 3. 01-agentic-rag/lessons/16-other-frameworks.md
 4. 01-agentic-rag/lessons/15-frameworks.md
 5. 01-agentic-rag/lessons/13-function-calling.md

KEYWORD SEARCH RESULTS
 1. 01-agentic-rag/lessons/14-agentic-loop.md
 2. 01-agentic-rag/lessons/13-function-calling.md
 3. 01-agentic-rag/lessons/13-function-calling.md
 4. 01-agentic-rag/lessons/13-function-calling.md
 5. 04-evaluation/lessons/02-ground-truth.md

Hybrid SEARCH RESULTS
 1. 01-agentic-rag/lessons/13-function-calling.md
 2. 01-agentic-rag/lessons/01-intro.md
 3. 01-agentic-rag/lessons/14-agentic-loop.md
 4. 04-evaluation/lessons/02-ground-truth.md
 5. 01-agentic-rag/lessons/16-other-frameworks.md
